## Homework 5: полное решение


## Подготовка окружения


```bash
uv sync --locked
sha256sum pipeline.bin
uv run python smoke_test.py
```

Контрольная сумма модели: `1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1`.


In [1]:
import hashlib
import json
from pathlib import Path
import tomllib

project_name = "ml-zoomcamp-homework-05-deployment-full"
candidates = [
    Path.cwd(),
    Path.cwd() / project_name,
    Path.home() / "Downloads" / project_name,
]
PROJECT = next(
    (path for path in candidates
     if path.name == project_name and (path / "evidence").is_dir()),
    None,
)
if PROJECT is None:
    raise FileNotFoundError("Open the notebook from the extracted project folder.")

standalone = json.loads((PROJECT / "evidence/standalone-evidence.json").read_text())
container = json.loads((PROJECT / "evidence/evidence.json").read_text())
metadata = json.loads((PROJECT / "model_metadata.json").read_text())

assert standalone["status"] == "passed"
assert container["status"] == "passed" and container["container_verified"]
model_sha = hashlib.sha256((PROJECT / "pipeline.bin").read_bytes()).hexdigest()
assert model_sha == metadata["model_sha256"]
assert model_sha == container["health"]["model_sha256"]
print("Model SHA256:", model_sha)
print("Smoke test:", standalone["official_smoke"])


Model SHA256: 1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1
Smoke test: {'conversion_probability': 0.769799, 'conversion': True}


## Question 1: uv

```bash
uv --version
```


In [2]:
answer_q1 = standalone["uv_version"]
print("uv on the server:", answer_q1)
print("uv in the Docker image:", container["uv_version"])


uv on the server: uv 0.11.11 (x86_64-unknown-linux-gnu)
uv in the Docker image: uv 0.10.11


## Question 2: версия Scikit-Learn


In [3]:
config = tomllib.loads((PROJECT / "pyproject.toml").read_text())
lock = tomllib.loads((PROJECT / "uv.lock").read_text())
answer_q2 = next(
    package["version"]
    for package in lock["package"]
    if package["name"] == "scikit-learn"
)
assert "scikit-learn==" + answer_q2 in config["project"]["dependencies"]
assert answer_q2 == metadata["scikit_learn"] == "1.7.2"
print("Answer Q2:", answer_q2)


Answer Q2: 1.7.2


## Question 3: загрузка модели

```bash
uv run python q3_predict.py
```

Расчёт внутри `q3_predict.py`:

```python
import json
import pickle
from pathlib import Path
from model import normalize_record

lead = json.loads(Path("lead-q3.json").read_text())
with open("pipeline.bin", "rb") as stream:
    pipeline = pickle.load(stream)

probability = float(pipeline.predict_proba([normalize_record(lead)])[0, 1])
print(format(probability, ".3f"))
```


In [4]:
result_q3 = standalone["q3"]
answer_q3 = result_q3["answer"]
assert answer_q3 == format(result_q3["conversion_probability"], ".3f")
print("Prediction:", result_q3)
print("Answer Q3:", answer_q3)


Prediction: {'conversion_probability': 0.5329944186432507, 'conversion': True, 'answer': '0.533'}
Answer Q3: 0.533


## Question 4: API без контейнера


In [5]:
result_q4 = standalone["q4"]
assert result_q4["health"]["status"] == "ok"
assert result_q4["health"]["model_sha256"] == model_sha
assert isinstance(result_q4["prediction"]["conversion"], bool)
answer_q4 = result_q4["answer"]
print("API without Docker:", result_q4["prediction"])
print("Health:", result_q4["health"])
print("Answer Q4:", answer_q4)


API without Docker: {'conversion_probability': 0.769799, 'conversion': True}
Health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
Answer Q4: 0.770


## Container: сборка и запуск


```bash
RUN_ID="hw05-$(date +%s)-$$"
IMAGE="zoomcamp-model:$RUN_ID"
BUILDER="$RUN_ID-builder"
CONTAINER="$RUN_ID-api"

BUILDKIT_BEFORE=$(docker image inspect moby/buildkit:buildx-stable-1 --format '{{.Id}}' 2>/dev/null || true)

docker buildx create --name "$BUILDER" --driver docker-container
docker buildx build --builder "$BUILDER" --load --tag "$IMAGE" .

docker run --detach --name "$CONTAINER" \
  --publish 127.0.0.1:9696:9696 \
  --memory 512m --cpus 1 \
  --cap-drop ALL --security-opt no-new-privileges:true \
  "$IMAGE"
```

Образ собирается по исходному `Dockerfile` с неизменённым `uv.lock`. `buildx build --load` выполняет сборку и загружает образ в Docker.


In [6]:
assert container["container_verified"]
assert container["container"]["readonly_rootfs"]
assert all(
    binding["HostIp"] == "127.0.0.1"
    for binding in container["container"]["bindings"]
)
print("Built image:", container["image"]["id"])
print("Architecture:", container["image"]["architecture"])
print("Image size, MB:", round(container["image"]["size_bytes"] / 1_000_000, 1))
print("Container ID:", container["container"]["id"])
print("Container actually started:", container["container_verified"])


Built image: sha256:7deb40b23c0576a4aacff514a9e8a90e42de3bba6d9d22df8155397a87ea62ea
Architecture: amd64
Image size, MB: 686.2
Container ID: c826c1a871274234b2eea5d49a812f6312dc4fe1909edbf1af7fe096b3ede872
Container actually started: True


## Question 5: базовый образ


In [7]:
dockerfile = (PROJECT / "Dockerfile").read_text()
base_image = next(
    line.split()[1]
    for line in dockerfile.splitlines()
    if line.startswith("FROM ")
)
answer_q5 = base_image.split("@", 1)[0]
assert answer_q5 == "python:3.11.15-slim-bookworm"
print("Answer Q5:", answer_q5)


Answer Q5: python:3.11.15-slim-bookworm


## Question 6: запрос к контейнеру

После старта API в контейнере:

```bash
uv run python q6_test.py
curl -s http://127.0.0.1:9696/health
```


In [8]:
result_q6 = container["q6"]
assert result_q6 == container["official_q6_smoke"]
assert result_q6 == container["q4"]
assert abs(
    result_q6["conversion_probability"]
    - result_q4["prediction"]["conversion_probability"]
) <= 0.005
assert container["health"]["status"] == "ok"
assert container["health"]["model_sha256"] == model_sha
assert container["health"]["feature_defaults_sha256"] == metadata["feature_defaults_sha256"]

answer_q6 = format(result_q6["conversion_probability"], ".3f")
print("Official q6_test.py:", container["official_q6_smoke"])
print("Container health:", container["health"])
print("Answer Q6:", answer_q6)


Official q6_test.py: {'conversion_probability': 0.769799, 'conversion': True}
Container health: {'status': 'ok', 'model_sha256': '1646bbdcd38d4f044da6b630c5b332c93a314245a8c21929011c42de51f629f1', 'feature_defaults_sha256': '1694f842b908fac9f53c0cc19be6d91f934befc63af23cc32fefafde69acd0cd'}
Answer Q6: 0.770


## Очистка после проверки

В том же терминале:

```bash
docker stop "$CONTAINER"
docker rm "$CONTAINER"
docker image rm "$IMAGE"
docker buildx rm "$BUILDER"

if [ -z "$BUILDKIT_BEFORE" ]; then
  BUILD_IMAGE=$(docker image inspect moby/buildkit:buildx-stable-1 --format '{{.Id}}' 2>/dev/null || true)
  if [ -n "$BUILD_IMAGE" ] && [ -z "$(docker ps -aq --filter "ancestor=$BUILD_IMAGE")" ]; then
    docker image rm moby/buildkit:buildx-stable-1
  fi
fi
```

Общая очистка `docker system prune` не используею. В выполненной проверке также удалены временное окружение Python, журналы и каталоги на сервере.


In [9]:
assert standalone["api_stopped"] and standalone["api_process_group_stopped"]
assert container["cleanup_verified"]
assert container["existing_containers_preserved"]
assert container["final_cleanup"]["remote_context_removed"]
print("Standalone API stopped:", standalone["api_stopped"])
print("Temporary Docker resources removed:", container["cleanup_verified"])
print("Existing containers preserved:", container["existing_containers_preserved"])


Standalone API stopped: True
Temporary Docker resources removed: True
Existing containers preserved: True
